# Python tools and the agent loop

## Goal

Define Python tools, invoke one directly, and inspect a model response that requests multiple tools.

## Setup

Select the **Python (LangChain OpenAI)** kernel and run cells from top to bottom.
`OPENAI_API_KEY` and `OPENAI_MODEL` are read only from this course's root `.env`.
Shared settings are in `lesson_support.py`: up to 1024 output tokens, a 30-second
HTTP timeout, no HTTP retries, and LangSmith tracing disabled.
`make_model("another-api-model-id")` creates a different model with the same key.
The default text settings are intended for `gpt-5.6-luna`.
Creating a model does not call the API; `invoke` and consuming `stream` make paid requests.


## Tool definition

In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model

model = make_model()
print("Model:", model.model_name)


In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [ ]:
tool1.invoke({"x": 467})

In [ ]:
@tool("small_db", description="Fast access to small db")
def small_db() -> str:
    _db = 'Anna has 2 flowers, Dmitriy 5 cars, Alex 10 flats'
    return _db

## Adding to agents

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1, small_db],
    system_prompt="Use both available tools to answer the user. Distinguish facts returned by the notes from your interpretation; the notes do not give asset values."
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Read our notes and compare the listed possessions. Then calculate the square root of 28.7296.")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response['messages'])

In [ ]:
# Tool calls belong to AIMessage; do not depend on a fixed message index.
for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        print(message.tool_calls)
